# DBChroma to store the embedded vector

vector search with postgress (pgVector): https://www.youtube.com/watch?v=0P54MFyz-mc&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv&index=26



Improvement::

"""
NOTE::: the ingestion part::  text_per_document = doc['question'] + " " + doc['answer']
can be imporoved, 
text = f"""
Question:
{doc["question"]}

Answer:
{doc["answer"]}
"""
Then Chroma will return:

Document(
    page_content="""
    Question:
    I just discovered the course. Can I still join?

    Answer:
    Yes, but if you want a certificate...
    """
)

ALSO: we can get the metadata

def build_context(self, docs):

    lines = []

    for doc in docs:
        lines.append(f"Section: {doc.metadata['section']}")
        lines.append(doc.page_content)
        lines.append("")

    return "\n".join(lines)

    

# PHASE 1: Store the documents in DBChroma

Documents -> chuncking -> embedding -> vector database (chroma)

In [ ]:
from pathlib import Path
from rich import print
import os
import sys
import numpy as np
from sqlitesearch import VectorSearchIndex


parent_directory = os.path.dirname(os.getcwd())
sys.path.append(parent_directory)

from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import AutoTokenizer
from langchain_core.documents import Document
from langchain_chroma import Chroma # to store and retrieve the vectors and chunk from a database


# PERSIST_DIRECTORY="/workspace/data/chroma_db" # storage location
PERSIST_DIRECTORY="chroma_db" # storage location


# 1. load the documents
print('---------------------------------------------------------------')
print('Step 1: load the data')
documents = load_faq_data()
print(f'The number of documents is: {len(documents)}')
# print(documents[0])


print('---------------------------------------------------------------')
print('Step 2: Get the useful data only from each documant')
# extract only the question and answer from each documents
texts = list()

for doc in documents:
    text_per_document = doc['question'] + " " + doc['answer']
    texts.append(text_per_document)
# texts = [doc["question"] + " " + doc["answer"] for doc in documents]
print(f'The number of text documents is: {len(texts)}')
# print(texts[0])

# """
#     NOTE::: the ingestion part::  text_per_document = doc['question'] + " " + doc['answer']
#     can be imporoved, 
#     text = f"""
#     Question:
#     {doc["question"]}

#     Answer:
#     {doc["answer"]}
#     """
#     Then Chroma will return:

#     Document(
#         page_content="""
#         Question:
#         I just discovered the course. Can I still join?

#         Answer:
#         Yes, but if you want a certificate...
#     """


# """
#     )

#     ALSO: we can get the metadata

#     def build_context(self, docs):

#         lines = []

#         for doc in docs:
#             lines.append(f"Section: {doc.metadata['section']}")
#             lines.append(doc.page_content)
#             lines.append("")

#         return "\n".join(lines)
# """



print('---------------------------------------------------------------')
print('Step 3: chunck the data')


# # NOTE:: we can also use:
# from gitsource import chunk_documents
# chunks = chunk_documents(documents, size=2000, step=1000)


# 3.1. Convert your list of text strings into a list of LangChain Document objects
# This keeps your 1,350 documents separate and un-merged.
langchain_docs = [Document(page_content=text) for text in texts]


# Note::: I changed the chunck_size to 5000 so that I get the same numbers of vectors and documents to overcome an error
# 3.2. Setup the Recursive Text Splitter
# It chunks your text smartly, keeping paragraphs/sentences together, and creates overlaps!
# NOTE::: the number of vectors need to be exactly as the number of documents if I use minsearch
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=5000,    # A high number ensures a single Q&A text never gets split up
    # chunk_size=500, # for improvement
    chunk_overlap=0,    # Zero overlap ensures no repeated data, just like your old loop
    # chunk_overlap=50, # for improvement
    # length_function=lambda text: len(tokenizer.encode(text)) # Counting tokens, # for improvement and by default length_function=len
    # separators=[r"Question \d+:", "\n\n", "\n", " "],# for improvement. note custome seperator= separators=["\n===\n", "\n\n", "\n", " "] but since I have only question and answer, I set it differently
    # is_separator_regex=True # for improvement 
)

# split the docs (chuncks)
texts_chuncks = text_splitter.split_documents(langchain_docs) # Note: we do not use  text_splitter.split_text(" ".join(texts))  since we have a list of document, each document will be splitted into chuncks, then langchain merges all the chunck

print(f"The number of chunks documents is: {len(texts_chuncks)}")
# print(texts_chuncks[0])


print('---------------------------------------------------------------')
print('Step 4: embed the data')
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-V2", # same model as in datatalk
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True} # this is for normalizing (sumation =1 of a vector)
)

# Note when I use ChromaDB then I do not need to do embedding by myself as before
# The chromaDB does:: open a folder, open a DB and create collection, and convert the text to vectores and stor them
# 3.2 convert chunck to embedding vector and store them beside the chuncked text in the ChromaDB
# store metadata, chuncks and embedding, and allows semantic similarity search
#  chroma.from_documents does: Open (or create) the database, Convert every chunk into an embedding, and Store the embeddings + original text + metadata
chromadb_vector_store = Chroma.from_documents(
    documents=texts_chuncks,
    embedding=embeddings,
    collection_name="datatalk_training",
    persist_directory=PERSIST_DIRECTORY
)

# to get the number of records,
get_number_of_records = chromadb_vector_store._collection.count()
# check how many recodes are stored
print(f'the number of records is: {get_number_of_records}')

---------------------------------------------------------------

Step 1: load the data

The number of documents is: 1350

---------------------------------------------------------------

Step 2: Get the useful data only from each documant

...

Step 3: chunck the data

The number of chunks documents is: 1350

---------------------------------------------------------------

Step 4: embed the data

the number of records is: 1350

# PHASE 2 (Retriever & RAG with DBChrome)

### 2.1 retrieve from ChromaDB using as_retriever 

In [2]:
# 4. create a retriever (we need to optimize the Recall and the Precision and this is not included here
retriever  = chromadb_vector_store.as_retriever(search_type="similarity",# mmr ((Maximum Marginal Relevance) then we search_kwargs={ "k": 5,"fetch_k"} fetch the top 20 and then select the top 5 that are not repeated, "similarity_score_threshold" to fetch all when the similarity is greter than threshold search_kwargs={ "score_threshold": 0.75}
                                                search_kwargs={"k":5, # get the highest similarity vectors between the input and the stored vectors, very common to use
                                                              })
# test the retriever
query = 'I just found out about the program, can I still sign up?' 
results = retriever.invoke(query)
results

[Document(id='49d8b470-7278-4370-83f8-9d36fe4a68d2', metadata={}, page_content='I just discovered the course. Can I still join? Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'),
 Document(id='9673716a-996e-4b57-9a4c-783a7ea93328', metadata={}, page_content='I just discovered the course. Can I still join? Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'),
 Document(id='685e6d47-d487-42c8-aaec-64bf0834c527', metadata={}, page_content='I just discovered the course. Can I still join? Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'),
 Document(id='7d1cba88-f917-4670-b4cc-189c9b6b00e2', metadata={}, page_content='How do I sign up? In the course [GitHub repository](http://mlzoomcamp.com), there’s a link to sign up. Here it is: [airtable.com](https://airtable.com/shryxwLd0COOEaqX

In [3]:
print(results[0])

Document(
    id='49d8b470-7278-4370-83f8-9d36fe4a68d2',
    metadata={},
    page_content='I just discovered the course. Can I still join? Yes, but if you want to receive a certificate, 
you need to submit your project while we’re still accepting submissions.'
)

### update the build_context function 

In [4]:
# Chang the build context since I get differnt result know:
##########################################################################
# Convert the SearchResult to Context to be understandable by LLM
##############################################################################
def build_context(retrived_documents):
    return "\n\n".join(
        doc.page_content
        for doc in retrived_documents
    )

build_context(results)

'I just discovered the course. Can I still join? Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\nI just discovered the course. Can I still join? Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\nI just discovered the course. Can I still join? Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\nHow do I sign up? In the course [GitHub repository](http://mlzoomcamp.com), there’s a link to sign up. Here it is: [airtable.com](https://airtable.com/shryxwLd0COOEaqXo)\n\nHow do I sign up? In the course [GitHub repository](http://mlzoomcamp.com), there’s a link to sign up. Here it is: [airtable.com](https://airtable.com/shryxwLd0COOEaqXo)'

## retrive query by DBChroma 

In [5]:
# first we use the vector_store that we pass it in the search method
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-V2", # same model as in datatalk
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True} # this is for normalizing (sumation =1 of a vector)
)

vector_store = Chroma(
    persist_directory="chroma_db",
    embedding_function=embeddings,
    collection_name="datatalk_training"
)

vector_store.similarity_search(
            query,
            k=5
        )

[Document(id='49d8b470-7278-4370-83f8-9d36fe4a68d2', metadata={}, page_content='I just discovered the course. Can I still join? Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'),
 Document(id='9673716a-996e-4b57-9a4c-783a7ea93328', metadata={}, page_content='I just discovered the course. Can I still join? Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'),
 Document(id='685e6d47-d487-42c8-aaec-64bf0834c527', metadata={}, page_content='I just discovered the course. Can I still join? Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'),
 Document(id='7d1cba88-f917-4670-b4cc-189c9b6b00e2', metadata={}, page_content='How do I sign up? In the course [GitHub repository](http://mlzoomcamp.com), there’s a link to sign up. Here it is: [airtable.com](https://airtable.com/shryxwLd0COOEaqX

In [6]:
# second we store a retriever
retriever = vector_store.as_retriever(
    search_kwargs={"k": 5}
)
retriever.invoke(query)

[Document(id='49d8b470-7278-4370-83f8-9d36fe4a68d2', metadata={}, page_content='I just discovered the course. Can I still join? Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'),
 Document(id='9673716a-996e-4b57-9a4c-783a7ea93328', metadata={}, page_content='I just discovered the course. Can I still join? Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'),
 Document(id='685e6d47-d487-42c8-aaec-64bf0834c527', metadata={}, page_content='I just discovered the course. Can I still join? Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'),
 Document(id='7d1cba88-f917-4670-b4cc-189c9b6b00e2', metadata={}, page_content='How do I sign up? In the course [GitHub repository](http://mlzoomcamp.com), there’s a link to sign up. Here it is: [airtable.com](https://airtable.com/shryxwLd0COOEaqX

### update RAGBaseClass

In [8]:
class RAGVectorDBChroma(RAGBase):
    def __init__(self, vector_store, **kwargs):
        super().__init__(**kwargs)
        
        self.vector_store = vector_store
        # OR:::
        # self.retriever = retriever

    def search(self, query):
#         retriever = vector_store.as_retriever(
#                       search_kwargs={"k": self.num_results}
#         )
#         retriever.invoke(query)

        return self.vector_store.similarity_search(
            query,
            k=self.num_results
        )
    def build_context(self, search_results):
         return "\n\n".join(
        doc.page_content
        for doc in search_results
    )


In [13]:
vector_store = Chroma(
    persist_directory="chroma_db",
    embedding_function=embeddings,
    collection_name="datatalk_training"
)

assistant = RAGVectorDBChroma(
    vector_store=vector_store
)

query = 'I just found out about the program, can I still sign up?' 
answer = assistant.rag(query)
answer

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

# For deleting the DBChroma

In [ ]:
import shutil
from pathlib import Path

PERSIST_DIRECTORY = Path("chroma_db")

if PERSIST_DIRECTORY.exists():
    shutil.rmtree(PERSIST_DIRECTORY)
    print("Chroma database deleted.")

PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'chroma_db\\07be3322-2340-4790-abab-3664debb881f\\data_level0.bin'